# 🎛️ Regional Prompting — a different prompt per region

Put **different prompts in different regions** of one image (training-free). Left vs right here; the block supports any bboxes.

**Setup:** Runtime → GPU (A100) · free [HF token](https://huggingface.co/settings/tokens) · accept [FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev).

### 1 · Install

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf

### 2 · Sign in

In [ ]:
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"   # avoid transient HF-hub read timeouts on big downloads
import torch
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()

### 3 · Load the model (~2–3 min first run)

In [ ]:
from diffusers import ModularPipeline
pipe = ModularPipeline.from_pretrained("remyxai/regional-prompting-flux-modular", trust_remote_code=True)
pipe.load_components(dtype=torch.bfloat16); pipe.to("cuda")
print("✅ ready:", type(pipe.blocks).__name__)  # RegionalPromptingFluxBlock

### 4 · Two regions, two prompts
Left half and right half get different prompts (edit them). `bbox` is normalized [x0,y0,x1,y1].

In [ ]:
#@title Generate { display-mode: "form" }
BASE_PROMPT = "two vehicles side by side on a seamless plain grey studio backdrop"  #@param {type:"string"}
LEFT_PROMPT = "a red vintage car"  #@param {type:"string"}
RIGHT_PROMPT = "a blue bicycle"  #@param {type:"string"}
share_base = False  #@param {type:"boolean"}
region_isolate_strength = 0  #@param {type:"slider", min:0, max:5, step:0.5}
seed = 0  #@param {type:"integer"}
import torch
from IPython.display import display
g=torch.Generator("cuda").manual_seed(int(seed))
img=pipe(base_prompt=BASE_PROMPT,
         regions=[{"prompt":LEFT_PROMPT,"bbox":[0.0,0.0,0.5,1.0]},
                  {"prompt":RIGHT_PROMPT,"bbox":[0.5,0.0,1.0,1.0]}],
         region_exclusive=(not share_base),                 # share_base=True -> both regions also see the base scene (kills the tone seam)
         region_isolate_strength=float(region_isolate_strength),  # keeps objects apart; lower if seam, raise if fusion
         height=1024, width=1024, num_inference_steps=28, generator=g).images[0]
img.save("result.png"); print(f"left: {LEFT_PROMPT} | right: {RIGHT_PROMPT}  (share_base={share_base}, isolate={region_isolate_strength})"); display(img)

### Download

In [ ]:
from google.colab import files
files.download("result.png")

---
[`remyxai/regional-prompting-flux-modular`](https://huggingface.co/remyxai/regional-prompting-flux-modular) · training-free · non-commercial.